# Cargar y Preprocesar datos

Comenzaremos cargando los datos de nuestro dataset

In [ ]:
import pandas as pd
import numpy as np
import math

# Definimos la ruta del archivo que mencionaste antes
path = "lichess_games.csv"

# Cargamos una muestra de 10,000 filas para empezar
# Esto permite que el desarrollo sea fluido
df = pd.read_csv(path, nrows=10000)

print("Carga inicial completada. Tamaño de la muestra:", df.shape)

Haremos un preprocesamiento de los campos Elo

In [ ]:
# 1. Reemplazar "?" por "0" en las columnas de Elo 
df['WhiteElo'] = df['WhiteElo'].replace('?', '0')
df['BlackElo'] = df['BlackElo'].replace('?', '0')

# 2. Convertir a tipo numérico para poder hacer cálculos
df['WhiteElo'] = df['WhiteElo'].astype(int)
df['BlackElo'] = df['BlackElo'].astype(int)

# Visualización rápida del preprocesamiento inicial 
print("Datos preprocesados (Elo convertido a int):")
print(df[['WhiteElo', 'BlackElo', 'TimeControl']].head())

Añadimos las dos columnas GameElo y EloDiff

In [ ]:
# Calcular GameElo: CEIL((WhiteElo + BlackElo) / 2) 
df['GameElo'] = np.ceil((df['WhiteElo'] + df['BlackElo']) / 2).astype(int)

# Calcular EloDiff: WhiteElo - BlackElo 
df['EloDiff'] = df['WhiteElo'] - df['BlackElo']

print("Columnas GameElo y EloDiff añadidas correctamente.")

Seleccionaremos las variables identificando las columnas numéricas y aplicando las diferentes transformaciones.

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# Seleccionamos las columnas numéricas para escalar
features_num = ['WhiteElo', 'BlackElo', 'MovesCount', 'GameElo', 'EloDiff']


# --- ORIGINALES ---
df_original = df[features_num + ['Result']].copy()

X = df_original[features_num]

# --- ESTANDARIZACIÓN (Media 0, Varianza 1) ---
scaler_std = StandardScaler()
df_estandarizado = pd.DataFrame(scaler_std.fit_transform(X), columns=features_num)
# Añadimos la clase de nuevo para tener el conjunto completo
df_estandarizado['Result'] = df_original['Result'].values 

# --- NORMALIZACIÓN (Rango [0, 1]) ---
scaler_norm = MinMaxScaler()
df_normalizado = pd.DataFrame(scaler_norm.fit_transform(X), columns=features_num)
df_normalizado['Result'] = df_original['Result'].values

print("Conjunto original (primeras filas): ")
print(df_original.head())

print("Conjunto Estandarizado (primeras filas):")
print(df_estandarizado.head())

print("\nConjunto Normalizado (primeras filas):")
print(df_normalizado.head())

# PCA

In [ ]:
from sklearn.decomposition import PCA

# Función auxiliar para aplicar PCA y mantener la estructura
def aplicar_pca(data, varianza, nombre_columnas):
    pca = PCA(n_components=varianza)
    # Aplicamos PCA solo a las características numéricas
    componentes = pca.fit_transform(data[nombre_columnas])
    
    # Creamos un DataFrame con los resultados
    df_pca = pd.DataFrame(
        data=componentes, 
        columns=[f'PC{i+1}' for i in range(componentes.shape[1])]
    )
    return df_pca

## PCA de 0.95

#### Datos originales

In [ ]:
# PCA 0.95 sobre el conjunto original
df_originalPCA95 = aplicar_pca(df_original, 0.95, features_num)
df_originalPCA95['Result'] = df_original['Result'].values

print(f"PCA 95% (Original) generó {df_originalPCA95.shape[1]} componentes.")

#### Datos estandarizados

In [ ]:
# PCA 0.95 sobre el conjunto estandarizado
df_estandarizadoPCA95 = aplicar_pca(df_estandarizado, 0.95, features_num)
df_estandarizadoPCA95['Result'] = df_estandarizado['Result'].values

print(f"PCA 95% (Estandarizado) generó {df_estandarizadoPCA95.shape[1]} componentes.")

#### Datos normalizados

In [ ]:
# PCA 0.80 sobre el conjunto normalizado
df_normalizadoPCA95 = aplicar_pca(df_normalizado, 0.95, features_num)
df_normalizadoPCA95['Result'] = df_normalizado['Result'].values

print(f"PCA 95% (Normalizado) generó {df_normalizadoPCA95.shape[1]} componentes.")

## PCA de 0.80

#### Datos originales

In [ ]:
# PCA 0.80 sobre el conjunto original
df_originalPCA80 = aplicar_pca(df_original, 0.80, features_num)
df_originalPCA80['Result'] = df_original['Result'].values

print(f"PCA 80% (Original) generó {df_originalPCA80.shape[1]} componentes.")

#### Datos estandarizados

In [ ]:
# PCA 0.80 sobre el conjunto estandarizado
df_estandarizadoPCA80 = aplicar_pca(df_estandarizado, 0.80, features_num)
df_estandarizadoPCA80['Result'] = df_estandarizado['Result'].values

print(f"PCA 80% (Estandarizado) generó {df_estandarizadoPCA80.shape[1]} componentes.")

#### Datos normalizados

In [ ]:
# PCA 0.80 sobre el conjunto normalizado
df_normalizadoPCA80 = aplicar_pca(df_normalizado, 0.80, features_num)
df_normalizadoPCA80['Result'] = df_normalizado['Result'].values

print(f"PCA 80% (Normalizado) generó {df_normalizadoPCA80.shape[1]} componentes.")

## Configurar la validación cruzada (k=5)

In [ ]:
from sklearn.model_selection import StratifiedKFold

# Configuramos k=5 iteraciones 
# Usamos Stratified para que el % de victorias/derrotas sea igual en cada parte
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Creamos los índices que usaremos para entrenar y validar
target = df_original['Result']
folds = list(skf.split(df_original[features_num], target))

print(f"Validación cruzada de {len(folds)} iteraciones configurada.")

Guardaremos los 5-Folds dividiendo cada una de las versiones en 5 partes para poder usarlas en `train.ipynb`

In [ ]:
import os
from sklearn.model_selection import StratifiedKFold

# Definir los 9 datasets
datasets_dict = {
    "original": df_original,
    "estandarizado": df_estandarizado,
    "normalizado": df_normalizado,
    "original_PCA95": df_originalPCA95,
    "original_PCA80": df_originalPCA80,
    "estandarizado_PCA95": df_estandarizadoPCA95,
    "estandarizado_PCA80": df_estandarizadoPCA80,
    "normalizado_PCA95": df_normalizadoPCA95,
    "normalizado_PCA80": df_normalizadoPCA80
}

# Configurar la Validación Cruzada (k=5)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
DATA_PATH = './kfolds_data'

# Bucle para generar las carpetas y los archivos CSV
for nombre, data in datasets_dict.items():
    folder_path = f"{DATA_PATH}/conj_{nombre}"
    os.makedirs(folder_path, exist_ok=True)
    
    X = data.drop('Result', axis=1)
    y = data['Result']
    
    # Dividir en 5 trozos
    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
        # Crear conjuntos de entrenamiento y validación para este fold
        train_df = data.iloc[train_idx]
        valid_df = data.iloc[val_idx]
        
        # Guardar con los nombres que espera tu train.ipynb
        train_df.to_csv(f"{folder_path}/training_{fold}.csv", index=False)
        valid_df.to_csv(f"{folder_path}/validating_{fold}.csv", index=False)

print("Se han generado todas las carpetas en ./kfolds_data")